# Лабораторная работа 4: Ансамбли и полносвязные нейронные сети

## Цель работы
Решить задачу классификации кредитоспособности клиентов с использованием:
- Ансамблевых методов (Random Forest, Gradient Boosting)
- Полносвязной нейронной сети (MLP)

## План выполнения
1. Загрузка и анализ данных
2. Предобработка и разделение выборки
3. Обучение базовых моделей (baseline)
4. Оптимизация гиперпараметров через GridSearchCV
5. Сравнение метрик на тестовой выборке

## Критерии оценки (ROC-AUC на тесте)
| ROC-AUC | Баллы |
|---------|-------|
| ≤ 0.76  | 0     |
| 0.76–0.77 | 2   |
| 0.77–0.78 | 4   |
| 0.78–0.79 | 6   |
| 0.79–0.80 | 8   |
| > 0.80  | 10    |

In [ ]:
# Импорт библиотек
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Модели машинного обучения
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

# Утилиты для подготовки данных и оценки
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score
)

# Настройки для воспроизводимости и визуализации
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# ============================================================
# ШАГ 1: Загрузка данных
# ============================================================

# Загружаем датасет (разделитель - точка с запятой)
data = pd.read_csv('german.csv', sep=';')

# Краткая информация о данных
print(f"Размер датасета: {data.shape[0]} строк, {data.shape[1]} столбцов")
print(f"\nЦелевая переменная: 'Creditability'")
print(f"Признаки: {list(data.columns[1:])}")

# Просмотр первых строк
print("\nПервые 5 записей:")
print(data.head())

# Проверка качества данных
print(f"\nПропущенные значения: {data.isnull().sum().sum()}")
print(f"Типы данных:\n{data.dtypes.value_counts()}")

In [ ]:
# ============================================================
# ШАГ 2: Разведочный анализ данных (EDA)
# ============================================================

target_column = 'Creditability'

# Распределение целевой переменной
class_counts = data[target_column].value_counts().sort_index()

plt.figure(figsize=(6, 4))
plt.bar([0, 1], class_counts.values, color=['#ff6b6b', '#4ecdc4'], edgecolor='black')
plt.xticks([0, 1], ['Ненадёжный (0)', 'Надёжный (1)'])
plt.xlabel('Класс')
plt.ylabel('Количество клиентов')
plt.title('Распределение целевой переменной')
plt.tight_layout()
plt.show()

# Статистика по классам
print(f"Доля надёжных клиентов: {data[target_column].mean():.2%}")
print(f"Распределение классов:\n{data[target_column].value_counts(normalize=True)}")

In [ ]:
# ============================================================
# ШАГ 3: Подготовка данных к обучению
# ============================================================

# Разделяем признаки и целевую переменную
X = data.drop(columns=[target_column]).values
y = data[target_column].values

# Разделяем на обучающую и тестовую выборки (80/20)
# stratify=y сохраняет пропорции классов в обеих выборках
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Обучающая выборка: {X_train.shape}")
print(f"Тестовая выборка: {X_test.shape}")
print(f"Классы в train: {np.bincount(y_train)}")
print(f"Классы в test: {np.bincount(y_test)}")

# Масштабирование признаков для нейронной сети (стандартизация)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# ============================================================
# ШАГ 4: Обучение базовых моделей (baseline)
# ============================================================

def evaluate_model(model, X_train, X_test, y_train, y_test, model_name, scaled=False):
    """
    Обучает модель и выводит метрики качества.
    
    Параметры:
    - model: экземпляр модели sklearn
    - scaled: использовать ли масштабированные данные (для MLP)
    """
    # Выбираем данные: масштабированные или исходные
    if scaled:
        X_tr = scaler.transform(X_train)
        X_te = scaler.transform(X_test)
    else:
        X_tr = X_train
        X_te = X_test
    
    # Обучаем модель
    model.fit(X_tr, y_train)
    
    # Получаем предсказания (вероятности для ROC-AUC)
    if hasattr(model, 'predict_proba'):
        y_pred_proba = model.predict_proba(X_te)[:, 1]
    else:
        # Для моделей без predict_proba нормализуем decision_function
        scores = model.decision_function(X_te)
        y_pred_proba = (scores - scores.min()) / (scores.max() - scores.min())
    
    y_pred = model.predict(X_te)
    
    # Считаем метрики
    metrics = {
        'ROC-AUC': roc_auc_score(y_test, y_pred_proba),
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred)
    }
    
    # Выводим результаты
    print(f"\n{model_name}:")
    for name, value in metrics.items():
        print(f"  {name}: {value:.4f}")
    
    return metrics, model


# 4.1 Random Forest (baseline параметры)
rf_baseline = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf_metrics_base, rf_model_base = evaluate_model(
    rf_baseline, X_train, X_test, y_train, y_test, "Random Forest (baseline)"
)

# 4.2 Gradient Boosting (baseline параметры)
gb_baseline = GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE)
gb_metrics_base, gb_model_base = evaluate_model(
    gb_baseline, X_train, X_test, y_train, y_test, "Gradient Boosting (baseline)"
)

# 4.3 MLP Neural Network (baseline параметры)
mlp_baseline = MLPClassifier(
    hidden_layer_sizes=(30,),
    max_iter=500,
    random_state=RANDOM_STATE,
    early_stopping=True
)
mlp_metrics_base, mlp_model_base = evaluate_model(
    mlp_baseline, X_train, X_test, y_train, y_test,
    "MLP Neural Network (baseline)",
    scaled=True
)

In [ ]:
# ============================================================
# ШАГ 5: Оптимизация гиперпараметров (GridSearchCV)
# ============================================================

print("Запуск подбора гиперпараметров...\n")


# 5.1 Random Forest: расширенный поиск параметров
print("Оптимизация Random Forest...")

rf_param_grid = {
    'n_estimators': [150, 200, 250],
    'max_depth': [8, 12, 16, None],
    'min_samples_split': [3, 5, 7],
    'min_samples_leaf': [1, 2, 3],
    'max_features': ['sqrt', 'log2'],
    'class_weight': ['balanced', None]
}

rf_grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid=rf_param_grid,
    cv=5,
    scoring='roc_auc',
    verbose=0,
    n_jobs=-1
)
rf_grid.fit(X_train, y_train)

print(f"Лучшие параметры: {rf_grid.best_params_}")
rf_metrics_opt, rf_model_opt = evaluate_model(
    rf_grid.best_estimator_, X_train, X_test, y_train, y_test,
    "Random Forest (optimized)"
)


# 5.2 Gradient Boosting: тонкая настройка
print("\nОптимизация Gradient Boosting...")

gb_param_grid = {
    'n_estimators': [150, 200],
    'learning_rate': [0.03, 0.05, 0.07],
    'max_depth': [3, 4, 5],
    'min_samples_split': [2, 4],
    'min_samples_leaf': [1, 2],
    'subsample': [0.8, 0.9],
    'max_features': ['sqrt', 'log2']
}

gb_grid = GridSearchCV(
    estimator=GradientBoostingClassifier(random_state=RANDOM_STATE),
    param_grid=gb_param_grid,
    cv=5,
    scoring='roc_auc',
    verbose=0,
    n_jobs=-1
)
gb_grid.fit(X_train, y_train)

print(f"Лучшие параметры: {gb_grid.best_params_}")
gb_metrics_opt, gb_model_opt = evaluate_model(
    gb_grid.best_estimator_, X_train, X_test, y_train, y_test,
    "Gradient Boosting (optimized)"
)


# 5.3 MLP Neural Network: архитектура и регуляризация
print("\nОптимизация MLP Neural Network...")

mlp_param_grid = {
    'hidden_layer_sizes': [(64,), (100,), (64, 32), (128, 64)],
    'activation': ['relu', 'tanh'],
    'alpha': [1e-4, 1e-3, 1e-2],  # L2-регуляризация
    'learning_rate_init': [0.001, 0.005],
    'max_iter': [800, 1000],
    'early_stopping': [True]
}

mlp_grid = GridSearchCV(
    estimator=MLPClassifier(random_state=RANDOM_STATE, early_stopping=True),
    param_grid=mlp_param_grid,
    cv=5,
    scoring='roc_auc',
    verbose=0,
    n_jobs=-1
)
mlp_grid.fit(X_train_scaled, y_train)

print(f"Лучшие параметры: {mlp_grid.best_params_}")
mlp_metrics_opt, mlp_model_opt = evaluate_model(
    mlp_grid.best_estimator_, X_train, X_test, y_train, y_test,
    "MLP Neural Network (optimized)",
    scaled=True
)

In [ ]:
# ============================================================
# ШАГ 6: Сравнение результатов всех моделей
# ============================================================

# Собираем все метрики в таблицу
results = pd.DataFrame({
    'Model': [
        'Random Forest (baseline)',
        'Random Forest (optimized)',
        'Gradient Boosting (baseline)',
        'Gradient Boosting (optimized)',
        'MLP Neural Network (baseline)',
        'MLP Neural Network (optimized)'
    ],
    'ROC-AUC': [
        rf_metrics_base['ROC-AUC'], rf_metrics_opt['ROC-AUC'],
        gb_metrics_base['ROC-AUC'], gb_metrics_opt['ROC-AUC'],
        mlp_metrics_base['ROC-AUC'], mlp_metrics_opt['ROC-AUC']
    ],
    'Accuracy': [
        rf_metrics_base['Accuracy'], rf_metrics_opt['Accuracy'],
        gb_metrics_base['Accuracy'], gb_metrics_opt['Accuracy'],
        mlp_metrics_base['Accuracy'], mlp_metrics_opt['Accuracy']
    ],
    'Precision': [
        rf_metrics_base['Precision'], rf_metrics_opt['Precision'],
        gb_metrics_base['Precision'], gb_metrics_opt['Precision'],
        mlp_metrics_base['Precision'], mlp_metrics_opt['Precision']
    ],
    'Recall': [
        rf_metrics_base['Recall'], rf_metrics_opt['Recall'],
        gb_metrics_base['Recall'], gb_metrics_opt['Recall'],
        mlp_metrics_base['Recall'], mlp_metrics_opt['Recall']
    ]
})

# Сортируем по ROC-AUC (лучшие модели сверху)
results = results.sort_values('ROC-AUC', ascending=False).reset_index(drop=True)

# Визуализация: сравнение ROC-AUC по моделям
plt.figure(figsize=(10, 6))
bars = plt.barh(results['Model'], results['ROC-AUC'], 
                color=['#ff6b6b' if 'baseline' in m else '#4ecdc4' for m in results['Model']],
                edgecolor='black')
plt.xlabel('ROC-AUC Score')
plt.title('Сравнение моделей по ROC-AUC')
plt.axvline(x=0.80, color='gold', linestyle='--', label='Цель: > 0.80')
plt.legend()
plt.tight_layout()
plt.show()

# Вывод итоговой таблицы с форматированием
print("\nИТОГОВЫЕ РЕЗУЛЬТАТЫ (отсортировано по ROC-AUC):")
print(results.to_string(index=False, float_format="%.4f"))

# Определяем лучшую модель и предполагаемую оценку
best_row = results.iloc[0]
best_auc = best_row['ROC-AUC']
best_model_name = best_row['Model']

if best_auc > 0.80:
    grade = 10
elif best_auc > 0.79:
    grade = 8
elif best_auc > 0.78:
    grade = 6
elif best_auc > 0.77:
    grade = 4
elif best_auc > 0.76:
    grade = 2
else:
    grade = 0

print(f"\nЛучшая модель: {best_model_name}")
print(f"ROC-AUC: {best_auc:.4f}")
print(f"Предполагаемая оценка: {grade}/10 баллов")

In [ ]:
# ============================================================
# ДОПОЛНИТЕЛЬНО: Анализ важности признаков (для лучшей модели)
# ============================================================

# Выбираем лучшую модель для анализа важности признаков
if 'Random Forest' in best_model_name:
    best_model = rf_model_opt
elif 'Gradient Boosting' in best_model_name:
    best_model = gb_model_opt
else:
    best_model = None

if best_model is not None:
    # Получаем важность признаков и сортируем по убыванию
    importances = best_model.feature_importances_
    feature_names = data.drop(columns=[target_column]).columns
    
    # Топ-10 наиболее важных признаков
    top_indices = np.argsort(importances)[::-1][:10]
    
    # Визуализация важности признаков
    plt.figure(figsize=(10, 6))
    plt.barh(range(10), importances[top_indices][::-1], 
             color='#6c5ce7', edgecolor='black')
    plt.yticks(range(10), [feature_names[i] for i in top_indices[::-1]])
    plt.xlabel('Важность признака')
    plt.title('Топ-10 наиболее значимых признаков')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    # Вывод топ-5 признаков с их важностью
    print("\nТоп-5 наиболее важных признаков:")
    for i in range(5):
        idx = top_indices[i]
        print(f"{i+1}. {feature_names[idx]}: {importances[idx]:.4f}")


# ============================================================
# РЕКОМЕНДАЦИИ для дальнейшего улучшения модели
# ============================================================

print("\nРекомендации для повышения качества:")
print("1. Попробовать ансамбль моделей (VotingClassifier, Stacking)")
print("2. Добавить feature engineering: полиномиальные признаки, взаимодействия")
print("3. Использовать методы борьбы с дисбалансом: SMOTE, class_weight")
print("4. Для MLP: экспериментировать с архитектурой (dropout, batch normalization)")
print("5. Провести кросс-валидацию с большим числом фолдов для стабильности оценки")